# Part 3 — The Gaussian: the noise we will use

_Rigorous Courses · Diffusion Models — Part 3 of 12_

**One bell curve, four algebra rules, and the square-completing trick behind every diffusion derivation**

This notebook puts every claim from the lesson under a numerical microscope: the anatomy of the density formula, the reparameterization line $x = \mu + \sigma z$, the scaling and sum rules, the complete-the-square identity, the product of two bell curves, and the 2-D isotropic Gaussian. Every rule gets a simulation and an `assert` — nothing is taken on faith.

---
This notebook accompanies the lesson. Run cells top to bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt
import math

rng = np.random.default_rng(0)

## The bell curve, term by term

The star of the whole course:

$$\mathcal{N}(x;\, \mu, \sigma^2) = \frac{1}{\sqrt{2\pi\sigma^2}}\, \exp\!\left(-\frac{(x-\mu)^2}{2\sigma^2}\right)$$

Two parts, two jobs. The **exponent** $-\frac{(x-\mu)^2}{2\sigma^2}$ shapes the curve: it penalizes squared distance from the center $\mu$, measured in units of the spread $\sigma$. The **normalizer** $\frac{1}{\sqrt{2\pi\sigma^2}}$ is a fixed number that scales the curve so its total area is exactly 1 — the requirement every legal density must meet (Part 2). We now verify both jobs with real numbers.

### Step 1 — Write the density as a function

One function, two knobs (`mu` and `sigma2`), mirroring the formula symbol for symbol. The lesson computed two values by hand: the standard bell at $x = 2$ should be about $0.0540$, and its peak should be $1/\sqrt{2\pi} \approx 0.3989$. Check both.

In [ ]:
def normal_pdf(x, mu, sigma2):
    normalizer = 1.0 / np.sqrt(2.0 * np.pi * sigma2)
    exponent = -((x - mu) ** 2) / (2.0 * sigma2)
    return normalizer * np.exp(exponent)


value_at_2 = normal_pdf(2.0, 0.0, 1.0)
peak_value = normal_pdf(0.0, 0.0, 1.0)

print(f"N(x=2; mu=0, var=1)  = {value_at_2:.4f}   (hand value from the lesson: 0.0540)")
print(f"peak of the standard bell = {peak_value:.4f}   (1/sqrt(2*pi) = 0.3989)")

assert abs(value_at_2 - 0.0540) < 0.001
assert abs(peak_value - 0.3989) < 0.001

### Step 2 — Check that the area under the curve is 1

The normalizer's entire job is to make the total area 1. We approximate the area with a **Riemann sum**: chop the axis into thin slices of width `dx`, treat the curve as flat over each slice, and add up slice areas — height times width. The window $[-10, 10]$ is 10 sigmas wide on each side, so the area we miss outside it is astronomically small.

In [ ]:
xs = np.linspace(-10.0, 10.0, 20001)
dx = xs[1] - xs[0]
area = np.sum(normal_pdf(xs, 0.0, 1.0)) * dx

print(f"area under the standard bell = {area:.8f}")

assert abs(area - 1.0) < 1e-5

### Step 3 — Slide the center $\mu$

Changing $\mu$ moves the bell left and right without changing its shape at all. Three centers, same spread.

In [ ]:
xs_plot = np.linspace(-6.0, 7.0, 600)

fig, ax = plt.subplots(figsize=(8, 4))
for mu in [-2.0, 0.0, 3.0]:
    ax.plot(xs_plot, normal_pdf(xs_plot, mu, 1.0), label=f"mu = {mu:.0f}, var = 1")
ax.set_xlabel("x")
ax.set_ylabel("density")
ax.set_title("Sliding the center: mu moves the bell, shape unchanged")
ax.legend()

plt.show()

### Step 4 — Stretch the spread $\sigma$, and see the two jobs

Changing $\sigma$ trades height for width. The annotations point at each term's job: the **normalizer** sets the peak height $\frac{1}{\sigma\sqrt{2\pi}}$ (narrow bell = tall bell, so the area stays 1), and the **exponent** controls how fast the flanks die off (in sigma-units, every bell falls to about 61% of its peak one $\sigma$ out). Note the $\sigma = 0.5$ peak is near $0.8$, and a $\sigma = 0.1$ bell would peak near $3.99$ — a density above 1 is perfectly legal.

In [ ]:
xs_plot = np.linspace(-6.0, 6.0, 600)

fig, ax = plt.subplots(figsize=(8, 4.5))
for sigma in [0.5, 1.0, 2.0]:
    curve = normal_pdf(xs_plot, 0.0, sigma ** 2)
    ax.plot(xs_plot, curve, label=f"sigma = {sigma}")

peak_05 = 1.0 / (0.5 * np.sqrt(2.0 * np.pi))
one_sigma_height = normal_pdf(1.0, 0.0, 1.0)
ax.annotate("normalizer's job:\npeak = 1/(sigma*sqrt(2pi))",
            xy=(0.0, peak_05), xytext=(1.6, 0.72),
            arrowprops=dict(arrowstyle="->"))
ax.annotate("exponent's job:\n61% of peak at 1 sigma out",
            xy=(1.0, one_sigma_height), xytext=(2.4, 0.42),
            arrowprops=dict(arrowstyle="->"))
ax.set_xlabel("x")
ax.set_ylabel("density")
ax.set_title("Stretching the spread: sigma trades height for width")
ax.legend()

plt.show()

### Step 5 — Measure the 68-95-99.7 rule

The lesson claims: about 68.3% of the probability sits within 1 sigma of the center, 95.4% within 2, and 99.7% within 3. These are areas, so we measure them with Riemann sums over $[-k, k]$.

In [ ]:
def area_within(k_sigmas):
    xs_band = np.linspace(-k_sigmas, k_sigmas, 4001)
    dx_band = xs_band[1] - xs_band[0]
    return np.sum(normal_pdf(xs_band, 0.0, 1.0)) * dx_band


within_1 = area_within(1.0)
within_2 = area_within(2.0)
within_3 = area_within(3.0)

print(f"area within 1 sigma: {within_1:.4f}   (rule says 0.683)")
print(f"area within 2 sigma: {within_2:.4f}   (rule says 0.954)")
print(f"area within 3 sigma: {within_3:.4f}   (rule says 0.997)")

assert abs(within_1 - 0.683) < 0.002
assert abs(within_2 - 0.954) < 0.002
assert abs(within_3 - 0.997) < 0.002

## Standardize and reparameterize

The lesson's most reused line — the **reparameterization** of a Gaussian:

$$x = \mu + \sigma z, \qquad z \sim \mathcal{N}(0, 1)$$

Draw a standard-normal $z$, stretch by $\sigma$, shift by $\mu$, and you have a draw from $\mathcal{N}(\mu, \sigma^2)$. Running it backwards is **standardization**: $z = (x - \mu)/\sigma$. We verify both directions.

### Step 6 — Build any Gaussian from standard draws

Target: $\mathcal{N}(2, 0.25)$, so $\mu = 2$ and $\sigma = 0.5$. We take 100,000 standard draws, apply the one-line recipe, and overlay the histogram on the target density. The histogram should land right on the curve, and the sample mean and standard deviation should match $\mu$ and $\sigma$.

In [ ]:
mu = 2.0
sigma = 0.5
n = 100000

z = rng.standard_normal(n)
x = mu + sigma * z

sample_mean = x.mean()
sample_std = x.std()

print(f"sample mean = {sample_mean:.4f}   (target mu = {mu})")
print(f"sample std  = {sample_std:.4f}   (target sigma = {sigma})")

assert abs(sample_mean - mu) < 0.01
assert abs(sample_std - sigma) < 0.01

xs_plot = np.linspace(0.0, 4.0, 400)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(x, bins=80, density=True, alpha=0.5, label="histogram of mu + sigma*z")
ax.plot(xs_plot, normal_pdf(xs_plot, mu, sigma ** 2), lw=2, label="target N(2, 0.25) density")
ax.set_xlabel("x")
ax.set_ylabel("density")
ax.set_title("Reparameterization: standard draws become N(2, 0.25) draws")
ax.legend()

plt.show()

### Step 7 — Standardize back, and re-run the lesson's hand example

Undo the recipe: $z = (x - \mu)/\sigma$ should hand back a standard normal — mean 0, standard deviation 1. Then the lesson's hand example: clean value $x_0 = 2$, noise spread $\sigma = 0.5$, draw $z = 1.2$, noisy value $2 + 0.5 \cdot 1.2 = 2.6$ — and standardizing $2.6$ must return exactly $1.2$.

In [ ]:
z_back = (x - mu) / sigma

print(f"standardized mean = {z_back.mean():.4f}   (target 0)")
print(f"standardized std  = {z_back.std():.4f}   (target 1)")

assert abs(z_back.mean()) < 0.02
assert abs(z_back.std() - 1.0) < 0.02

x0 = 2.0
z_draw = 1.2
noisy = x0 + 0.5 * z_draw
round_trip = (noisy - x0) / 0.5

print(f"\nhand example: noisy value = {noisy:.1f}   (lesson says 2.6)")
print(f"hand example: round trip  = {round_trip:.1f}   (back to z = 1.2)")

assert abs(noisy - 2.6) < 1e-12
assert abs(round_trip - 1.2) < 1e-12

## Scaling and adding Gaussians

Two algebra rules, both derived in the lesson:

- **Scaling rule:** $X \sim \mathcal{N}(\mu, \sigma^2) \Rightarrow aX \sim \mathcal{N}(a\mu,\, a^2\sigma^2)$ — variance picks up $a$ **squared**.
- **Sum rule:** for **independent** $X \sim \mathcal{N}(\mu_1, \sigma_1^2)$ and $Y \sim \mathcal{N}(\mu_2, \sigma_2^2)$: $X + Y \sim \mathcal{N}(\mu_1 + \mu_2,\, \sigma_1^2 + \sigma_2^2)$.

The means and variances follow from Part 2's rules. The claim that the *shape* stays Gaussian is the special part — the lesson states it without proof, so here the simulation carries the burden of evidence.

### Step 8 — Verify the scaling rule

Take $X \sim \mathcal{N}(1, 4)$ and multiply by $a = 3$. The rule predicts $3X \sim \mathcal{N}(3, 36)$: mean times 3, variance times 9. Watch the variance — it must jump by $a^2 = 9$, not by $a = 3$.

In [ ]:
a = 3.0
x_scale = 1.0 + 2.0 * rng.standard_normal(200000)
scaled = a * x_scale

scaled_mean = scaled.mean()
scaled_var = scaled.var()

print(f"mean of 3X = {scaled_mean:.3f}   (predicted 3.0)")
print(f"var  of 3X = {scaled_var:.3f}   (predicted 36.0 = 9 x 4, NOT 12 = 3 x 4)")

assert abs(scaled_mean - 3.0) < 0.05
assert abs(scaled_var - 36.0) < 0.5

### Step 9 — Verify the sum rule, including the Gaussian shape

Independent $X \sim \mathcal{N}(1, 4)$ and $Y \sim \mathcal{N}(2, 9)$. Prediction: $X + Y \sim \mathcal{N}(3, 13)$. We check the mean and variance with asserts, then check the *shape* claim: the histogram of the sum should lie on the predicted $\mathcal{N}(3, 13)$ curve everywhere, not merely share its mean and variance. Note the standard deviation $\sqrt{13} \approx 3.61$ — not $2 + 3 = 5$. Sigmas do not add; variances do.

In [ ]:
x_sum = 1.0 + 2.0 * rng.standard_normal(200000)
y_sum = 2.0 + 3.0 * rng.standard_normal(200000)
s = x_sum + y_sum

s_mean = s.mean()
s_var = s.var()

print(f"mean of X+Y = {s_mean:.3f}   (predicted 3.0)")
print(f"var  of X+Y = {s_var:.3f}   (predicted 13.0)")
print(f"std  of X+Y = {s.std():.3f}   (predicted sqrt(13) = 3.606, NOT 2 + 3 = 5)")

assert abs(s_mean - 3.0) < 0.05
assert abs(s_var - 13.0) < 0.2

hist_density, bin_edges = np.histogram(s, bins=60, range=(-12.0, 18.0), density=True)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
predicted_density = normal_pdf(bin_centers, 3.0, 13.0)
max_shape_gap = np.max(np.abs(hist_density - predicted_density))

print(f"\nlargest histogram-vs-curve gap = {max_shape_gap:.5f}  (curve peaks at 0.111)")

assert max_shape_gap < 0.01

xs_plot = np.linspace(-12.0, 18.0, 600)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(s, bins=60, range=(-12.0, 18.0), density=True, alpha=0.5, label="histogram of X + Y")
ax.plot(xs_plot, normal_pdf(xs_plot, 3.0, 13.0), lw=2, label="predicted N(3, 13) density")
ax.set_xlabel("x + y")
ax.set_ylabel("density")
ax.set_title("Sum of independent Gaussians: still exactly a bell")
ax.legend()

plt.show()

## Complete the square, checked by computer

The lesson's four-step recipe (collect, factor, add-and-subtract, regroup) claimed:

$$3x^2 - 12x + 7 \;=\; 3(x-2)^2 - 5$$

An algebraic identity must hold at *every* $x$, so we test it at a thousand points. The lesson's **Gaussian reading rule** — an exponent $-\frac{1}{2}(Ax^2 - 2Bx) + \text{const}$ hides a bell with variance $\frac{1}{A}$ and mean $\frac{B}{A}$ — gets its workout in the next section.

### Step 10 — Test the worked identity at a thousand points

If the two sides ever disagree, the recipe was run wrong. Floating-point arithmetic should agree to around $10^{-12}$.

In [ ]:
xs_check = np.linspace(-5.0, 5.0, 1001)
left_side = 3.0 * xs_check ** 2 - 12.0 * xs_check + 7.0
right_side = 3.0 * (xs_check - 2.0) ** 2 - 5.0
max_identity_gap = np.max(np.abs(left_side - right_side))

print(f"largest |left - right| over 1001 points = {max_identity_gap:.2e}")

assert max_identity_gap < 1e-9

## Multiplying two bell curves

The tool Part 7 runs on. Multiply two Gaussian densities in the same variable and renormalize — the result is another Gaussian:

$$\frac{1}{\sigma_*^2} = \frac{1}{\sigma_1^2} + \frac{1}{\sigma_2^2}, \qquad \mu_* = \sigma_*^2\left(\frac{\mu_1}{\sigma_1^2} + \frac{\mu_2}{\sigma_2^2}\right)$$

**Precisions add; the new center is the precision-weighted average.** The lesson's worked example: $\mathcal{N}(x;\, 1, 1) \cdot \mathcal{N}(x;\, 3, 2)$ should give $\sigma_*^2 = \frac{2}{3} \approx 0.667$ and $\mu_* = \frac{5}{3} \approx 1.667$ — between the centers but closer to the sharper bell.

### Step 11 — Multiply pointwise, renormalize, and compare

We multiply the two density arrays value by value, renormalize the product with a Riemann sum (the product alone has area below 1 — that is the $\propto$ in the lesson), and compare against the predicted bell at every grid point.

In [ ]:
grid = np.linspace(-4.0, 8.0, 4001)
dgrid = grid[1] - grid[0]

f1 = normal_pdf(grid, 1.0, 1.0)
f2 = normal_pdf(grid, 3.0, 2.0)
product_raw = f1 * f2
product_area = np.sum(product_raw) * dgrid
product_normalized = product_raw / product_area

precision_sum = 1.0 / 1.0 + 1.0 / 2.0
sigma2_star = 1.0 / precision_sum
mu_star = sigma2_star * (1.0 / 1.0 + 3.0 / 2.0)
predicted = normal_pdf(grid, mu_star, sigma2_star)

max_dev = np.max(np.abs(product_normalized - predicted))

print(f"raw product area (before renormalizing) = {product_area:.4f}  — below 1, as promised")
print(f"predicted mu_star     = {mu_star:.4f}   (lesson: 5/3 = 1.6667)")
print(f"predicted sigma2_star = {sigma2_star:.4f}   (lesson: 2/3 = 0.6667)")
print(f"largest pointwise deviation = {max_dev:.2e}")

assert abs(mu_star - 5.0 / 3.0) < 1e-12
assert abs(sigma2_star - 2.0 / 3.0) < 1e-12
assert max_dev < 1e-3

### Step 12 — See all three curves

Read the picture with the lesson's sanity checks: the product bell sits *between* the two inputs but closer to the sharper $\mathcal{N}(1,1)$, and it is *narrower and taller* than either input — multiplying evidence sharpens the answer.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(grid, f1, label="N(1, 1) — the sharper witness", color="#4ea1ff")
ax.plot(grid, f2, label="N(3, 2) — the vaguer witness", color="#ff7b72")
ax.plot(grid, product_normalized, lw=2.5, color="#3fb950", label="renormalized product")
ax.plot(grid, predicted, "--", lw=1.5, color="black", label="predicted N(5/3, 2/3)")
ax.set_xlabel("x")
ax.set_ylabel("density")
ax.set_title("Product of two bells = a sharper bell at the precision-weighted center")
ax.legend()

plt.show()

## Gaussians in many dimensions

The isotropic Gaussian $\mathcal{N}(\mu, \sigma^2\mathbf{I})$ is an independent 1-D bell in every coordinate:

$$\mathcal{N}(x;\, \mu, \sigma^2 \mathbf{I}) = \prod_{i=1}^{d} \mathcal{N}(x_i;\, \mu_i, \sigma^2)$$

In 2-D the equal-density contours are perfect circles (the merged exponent $-\frac{1}{2}(x_1^2 + x_2^2)$ only cares about distance from the center), and the two coordinates should be uncorrelated. This is the $\epsilon \sim \mathcal{N}(0, \mathbf{I})$ of the notation contract — one independent standard bell per pixel.

### Step 13 — Scatter the 2-D standard Gaussian with its circular contours

4,000 draws from $\mathcal{N}(0, \mathbf{I})$ in 2-D, contours of the product density on top, and a correlation check: with independent coordinates, the correlation should be near 0 (sampling noise of roughly $\pm 1/\sqrt{4000} \approx 0.016$).

In [ ]:
n2 = 4000
points = rng.standard_normal((n2, 2))

corr = np.corrcoef(points[:, 0], points[:, 1])[0, 1]

print(f"correlation between the two coordinates = {corr:.4f}   (independent => near 0)")

assert abs(corr) < 0.05

grid_1d = np.linspace(-4.0, 4.0, 200)
gx, gy = np.meshgrid(grid_1d, grid_1d)
density_2d = normal_pdf(gx, 0.0, 1.0) * normal_pdf(gy, 0.0, 1.0)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(points[:, 0], points[:, 1], s=4, alpha=0.3, label="4,000 draws from N(0, I)")
ax.contour(gx, gy, density_2d, levels=6, colors="black", linewidths=0.8)
ax.set_xlabel("coordinate 1")
ax.set_ylabel("coordinate 2")
ax.set_title("Isotropic 2-D Gaussian: circular blob, circular contours")
ax.set_aspect("equal")
ax.legend(loc="upper right")

plt.show()

## Practice

The same problems as the lesson. Try each one in the empty cell below it — on paper first where the problem asks for hand algebra — then reveal the worked solution.

**Problem 1.** Compute the Gaussian density $\mathcal{N}(x;\, \mu, \sigma^2)$ at $x = 1$ for $\mu = 0$ and $\sigma^2 = 4$, by hand to three decimals. Then check yourself with `normal_pdf`.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Normalizer: $\frac{1}{\sqrt{2\pi \cdot 4}} = \frac{1}{\sqrt{8\pi}} \approx 0.1995$.
- Exponent: $-\frac{(1-0)^2}{2 \cdot 4} = -0.125$, and $e^{-0.125} \approx 0.8825$.
- Product: $0.1995 \times 0.8825 \approx 0.176$.

```python
print(normal_pdf(1.0, 0.0, 4.0))   # 0.17603...
```

**Answer:** $\approx 0.176$ — well below the standard bell's peak of $0.399$, because a wide bell must be low to keep its area at 1.

</details>

**Problem 2.** An observation $x = 7.5$ came from $\mathcal{N}(5, 4)$. Standardize it and say, using the 68-95-99.7 rule, how surprising it is.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- $\sigma = \sqrt{4} = 2$ (standardize with the standard deviation, not the variance).
- Distance from center: $7.5 - 5 = 2.5$. In sigma-units: $z = 2.5 / 2 = 1.25$.
- $|z| = 1.25$ is beyond the 1-sigma band (68%) but well inside the 2-sigma band (95%).

```python
z_score = (7.5 - 5.0) / 2.0
print(z_score)   # 1.25
```

**Answer:** $z = 1.25$ — mildly unusual; roughly one draw in five lands further from the center than this.

</details>

**Problem 3.** $X \sim \mathcal{N}(2, 3)$ and $Y \sim \mathcal{N}(-1, 5)$ are independent. What is the distribution of $X + Y$? Give its standard deviation too — then verify by simulation.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Means add (linearity, Part 2): $2 + (-1) = 1$.
- Variances add (independence, Part 2): $3 + 5 = 8$.
- The shape stays Gaussian (the sum rule's special claim), so $X + Y \sim \mathcal{N}(1, 8)$, with standard deviation $\sqrt{8} \approx 2.83$ — not $\sqrt{3} + \sqrt{5} \approx 3.97$.

```python
xp = 2.0 + np.sqrt(3.0) * rng.standard_normal(200000)
yp = -1.0 + np.sqrt(5.0) * rng.standard_normal(200000)
sp = xp + yp
print(sp.mean(), sp.var())   # about 1 and about 8
```

**Answer:** $X + Y \sim \mathcal{N}(1, 8)$, standard deviation $\approx 2.83$.

</details>

**Problem 4.** $X \sim \mathcal{N}(4, 9)$. Find the distribution of $0.5X$ and of $-2X$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- $0.5X$: mean $0.5 \times 4 = 2$; variance $0.5^2 \times 9 = 2.25$. So $0.5X \sim \mathcal{N}(2, 2.25)$.
- $-2X$: mean $-2 \times 4 = -8$; variance $(-2)^2 \times 9 = 36$ — the square erases the sign, so the variance stays positive.

```python
xq = 4.0 + 3.0 * rng.standard_normal(200000)
print((0.5 * xq).mean(), (0.5 * xq).var())    # about 2 and 2.25
print((-2.0 * xq).mean(), (-2.0 * xq).var())  # about -8 and 36
```

**Answer:** $0.5X \sim \mathcal{N}(2, 2.25)$ and $-2X \sim \mathcal{N}(-8, 36)$.

</details>

**Problem 5.** Complete the square on the exponent $-\frac{1}{2}\left(3x^2 - 4x + 7\right)$ and read off the mean and variance of the bell it hides. (Paper first — then verify the identity numerically.)

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

Work inside the parentheses with the four-step recipe:

- **Collect:** $3x^2 - 4x$, constant $7$ set aside.
- **Factor:** $3\left(x^2 - \frac{4}{3}x\right)$.
- **Add and subtract:** half of $\frac{4}{3}$ is $\frac{2}{3}$, squared is $\frac{4}{9}$: $\;3\left(x^2 - \frac{4}{3}x + \frac{4}{9} - \frac{4}{9}\right)$.
- **Regroup:** $3\left(x - \frac{2}{3}\right)^2 - \frac{4}{3}$; restore the 7: $\;3\left(x - \frac{2}{3}\right)^2 + \frac{17}{3}$.

Apply the outer $-\frac{1}{2}$: the exponent is $-\frac{3}{2}\left(x - \frac{2}{3}\right)^2 - \frac{17}{6}$. Match against $-\frac{(x-\mu)^2}{2\sigma^2}$: from $\frac{1}{2\sigma^2} = \frac{3}{2}$, $\sigma^2 = \frac{1}{3}$; the mean is $\mu = \frac{2}{3}$.

```python
xs_p5 = np.linspace(-5.0, 5.0, 1001)
lhs = -0.5 * (3.0 * xs_p5 ** 2 - 4.0 * xs_p5 + 7.0)
rhs = -1.5 * (xs_p5 - 2.0 / 3.0) ** 2 - 17.0 / 6.0
print(np.max(np.abs(lhs - rhs)))   # ~1e-13
```

**Answer:** a Gaussian with mean $\frac{2}{3}$ and variance $\frac{1}{3}$ (times a constant).

</details>

**Problem 6.** Multiply the densities $\mathcal{N}(x;\, 0, 4)$ and $\mathcal{N}(x;\, 2, 4)$. Find the new mean and variance by the product rule, sanity-check them, then confirm numerically the way Step 11 did.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Precisions add: $\frac{1}{4} + \frac{1}{4} = \frac{1}{2}$, so $\sigma_*^2 = 2$.
- Weighted centers: $\frac{0}{4} + \frac{2}{4} = \frac{1}{2}$, so $\mu_* = 2 \times \frac{1}{2} = 1$.
- Sanity: equal spreads, so the center is the midpoint of 0 and 2 (it is), and the variance halves from 4 to 2 (it does).

```python
grid_p6 = np.linspace(-8.0, 10.0, 4001)
dg = grid_p6[1] - grid_p6[0]
prod = normal_pdf(grid_p6, 0.0, 4.0) * normal_pdf(grid_p6, 2.0, 4.0)
prod = prod / (np.sum(prod) * dg)
print(np.max(np.abs(prod - normal_pdf(grid_p6, 1.0, 2.0))))   # tiny
```

**Answer:** the product is proportional to $\mathcal{N}(x;\, 1, 2)$.

</details>

**Problem 7.** Which is more surprising: observing $x = 2.1$ from $\mathcal{N}(0, 1)$, or observing $x = 6$ from $\mathcal{N}(0, 9)$? Compute both z-scores *and* both density heights, and explain why they disagree.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- z-scores: $z_1 = 2.1 / 1 = 2.1$ and $z_2 = 6 / 3 = 2.0$. The first observation is further out in sigma-units.
- Densities: $\mathcal{N}(2.1;\, 0, 1) \approx 0.044$ but $\mathcal{N}(6;\, 0, 9) \approx 0.018$ — the densities point the *other* way.
- Resolution: the wide bell is low *everywhere* (its normalizer divides by $\sigma = 3$), so raw heights across bells with different spreads are not comparable. The standardized distance is the honest yardstick, and $2.1 > 2.0$.

```python
print(normal_pdf(2.1, 0.0, 1.0))   # 0.0440
print(normal_pdf(6.0, 0.0, 9.0))   # 0.0180
```

**Answer:** $x = 2.1$ under $\mathcal{N}(0, 1)$ is more surprising — standardize first, never compare raw heights.

</details>

## Wrap-up

Verified today, with real numbers: the normalizer really does make the area 1 and set the peak at $\frac{1}{\sigma\sqrt{2\pi}}$; the 68-95-99.7 areas check out; $\mu + \sigma z$ produces exactly the target bell and standardization undoes it; scaling multiplies the variance by $a^2$; the sum of independent Gaussians has the predicted mean and variance *and* the predicted bell shape; the complete-the-square identity holds to machine precision; the product of two bells is the precision-weighted bell to within $10^{-5}$; and the 2-D isotropic Gaussian is a circular cloud of independent coordinates.

Next, Part 4 sets these rules in motion: chains of random steps, and the first proof that the noising chain $x_t = \sqrt{1-\beta}\,x_{t-1} + \sqrt{\beta}\,\epsilon_t$ settles into exactly the standard bell you now know inside out.